In [6]:
import pandas as pd
from pathlib import Path
import openmeteo_requests
import requests_cache
from retry_requests import retry


def historical_forecast_to_parquet(
    responses,
    hourly_path="../data/processed/swiss_weather_historical_forecast_hourly.parquet",
    daily_path="../data/processed/swiss_weather_historical_forecast_daily.parquet",
):
    """
    Convert Open-Meteo historical forecast responses
    to hourly + daily parquet files.
    """

    cities = [
        "Zurich", "Geneva", "Bern", "Basel", "Lausanne",
        "Lucerne", "St_Gallen", "Lugano", "Interlaken", "Central_CH"
    ]

    hourly_dfs = []
    daily_dfs = []

    for i, response in enumerate(responses):

        # -------------------------
        # HOURLY
        # -------------------------
        hourly = response.Hourly()

        hourly_df = pd.DataFrame(
            {
                "timestamp_utc": pd.date_range(
                    start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
                    end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
                    freq=pd.Timedelta(seconds=hourly.Interval()),
                    inclusive="left",
                ),
                "temperature_2m": hourly.Variables(0).ValuesAsNumpy(),
                "relative_humidity_2m": hourly.Variables(1).ValuesAsNumpy(),
                "precipitation_probability": hourly.Variables(2).ValuesAsNumpy(),
                "rain": hourly.Variables(3).ValuesAsNumpy(),
                "snowfall": hourly.Variables(4).ValuesAsNumpy(),
                "snow_depth": hourly.Variables(5).ValuesAsNumpy(),
                "cloud_cover": hourly.Variables(6).ValuesAsNumpy(),
                "wind_speed_10m": hourly.Variables(7).ValuesAsNumpy(),
                "surface_pressure": hourly.Variables(8).ValuesAsNumpy(),
                "is_day": hourly.Variables(9).ValuesAsNumpy(),
                "sunshine_duration": hourly.Variables(10).ValuesAsNumpy(),
                "city": cities[i],
                "location_id": i,
                "latitude": response.Latitude(),
                "longitude": response.Longitude(),
                "elevation": response.Elevation(),
            }
        )

        hourly_dfs.append(hourly_df)

        # -------------------------
        # DAILY
        # -------------------------
        daily = response.Daily()

        daily_df = pd.DataFrame(
            {
                "date_utc": pd.date_range(
                    start=pd.to_datetime(daily.Time(), unit="s", utc=True),
                    end=pd.to_datetime(daily.TimeEnd(), unit="s", utc=True),
                    freq=pd.Timedelta(seconds=daily.Interval()),
                    inclusive="left",
                ),
                "temperature_2m_mean": daily.Variables(0).ValuesAsNumpy(),
                "daylight_duration": daily.Variables(1).ValuesAsNumpy(),
                "temperature_2m_min": daily.Variables(2).ValuesAsNumpy(),
                "temperature_2m_max": daily.Variables(3).ValuesAsNumpy(),
                "wind_gusts_10m_mean": daily.Variables(4).ValuesAsNumpy(),
                "wind_speed_10m_mean": daily.Variables(5).ValuesAsNumpy(),
                "city": cities[i],
                "location_id": i,
                "latitude": response.Latitude(),
                "longitude": response.Longitude(),
                "elevation": response.Elevation(),
            }
        )

        daily_dfs.append(daily_df)

    hourly_full = pd.concat(hourly_dfs, ignore_index=True)
    daily_full = pd.concat(daily_dfs, ignore_index=True)

    Path(hourly_path).parent.mkdir(parents=True, exist_ok=True)

    hourly_full.to_parquet(hourly_path, index=False)
    daily_full.to_parquet(daily_path, index=False)

    print(f"Saved: {hourly_path}")
    print(f"Saved: {daily_path}")

    return hourly_full, daily_full


# ======================================
# API CALL
# ======================================

cache_session = requests_cache.CachedSession(".cache", expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

url = "https://historical-forecast-api.open-meteo.com/v1/forecast"

params = {
    "latitude": [47.3769, 46.2044, 46.9480, 47.5596, 46.5197,
                 47.0502, 47.4245, 46.0101, 46.6863, 46.8182],

    "longitude": [8.5417, 6.1432, 7.4474, 7.5886, 6.6323,
                  8.3093, 9.3767, 8.9600, 7.8632, 8.2275],

    "start_date": "2020-01-01",
    "end_date": "2026-04-17",

    "hourly": [
        "temperature_2m",
        "relative_humidity_2m",
        "precipitation_probability",
        "rain",
        "snowfall",
        "snow_depth",
        "cloud_cover",
        "wind_speed_10m",
        "surface_pressure",
        "is_day",
        "sunshine_duration",
    ],

    "daily": [
        "temperature_2m_mean",
        "daylight_duration",
        "temperature_2m_min",
        "temperature_2m_max",
        "wind_gusts_10m_mean",
        "wind_speed_10m_mean",
    ],

    "timezone": "Europe/Zurich",
    "models": "best_match",
}

responses = openmeteo.weather_api(url, params=params)

hourly_df, daily_df = historical_forecast_to_parquet(responses)

Saved: ../data/processed/swiss_weather_historical_forecast_hourly.parquet
Saved: ../data/processed/swiss_weather_historical_forecast_daily.parquet
